<font size="6" color='grey'> <b>

Generative KI. Verstehen. Anwenden. Gestalten.
</b></font> </br>

<font size="5" color='grey'> <b>
M09 - Aufgabe A3: YOLO + MediaPipe Selfie Segmenter → Präzisions-Maske → Comic
</b></font> </br>

---

# Aufgabenbeschreibung

**Hybrid-Workflow mit maximaler Präzision:**

1. **YOLO Object Detection**: Findet Objekte und gibt Koordinaten
2. **MediaPipe Selfie Segmenter**: Erstellt Pixel-genaue Masken *innerhalb* der YOLO-Koordinaten
3. **Comic-Effekt**: Wende Comic-Filter mit der präzisen Maske an

**Vorteil gegenüber A2**: Pixel-genaue Segmentierung statt nur Rechteck-Masken!

---

# 1 | Umgebung einrichten

In [ ]:
#@title 🔧 Umgebung einrichten { display-mode: "form" }
!uv pip install --system -q git+https://github.com/ralf-42/GenAI.git#subdirectory=04_modul
from genai_lib.utilities import check_environment, get_ipinfo, setup_api_keys, mprint, install_packages
setup_api_keys(['OPENAI_API_KEY', 'HF_TOKEN'], create_globals=False)
print()
check_environment()
print()
get_ipinfo()

In [ ]:
#@title 🛠️ Installationen { display-mode: "form" }
install_packages([
    'ultralytics>=8.0.0',
    'mediapipe>=0.10.0',
    'opencv-python>=4.8.0',
    'pillow>=10.0.0'
])

In [ ]:
#@title 📂 Testbilder herunterladen { display-mode: "form" }
!rm -rf files
!mkdir -p files

!curl -L https://raw.githubusercontent.com/ralf-42/GenAI/main/02_daten/02_bild/peoples.png -o files/peoples.png
!curl -L https://raw.githubusercontent.com/ralf-42/GenAI/main/02_daten/02_bild/apfel.png -o files/apfel.png

print("✅ Testbilder heruntergeladen")

# 2 | Imports & Setup

In [ ]:
import cv2
import numpy as np
import mediapipe as mp
from ultralytics import YOLO
from PIL import Image as PILImage
from IPython.display import display, Image as IPImage
import matplotlib.pyplot as plt
import os

print("✅ Imports erfolgreich")

# 3 | YOLO + MediaPipe Initialization

In [ ]:
# YOLO Modell laden
print("📥 Lade YOLO v8 Modell...")
yolo_model = YOLO('yolov8m.pt')
print("✅ YOLO geladen")

# MediaPipe Selfie Segmenter initialisieren
print("📥 Lade MediaPipe Selfie Segmenter...")

# Download the MediaPipe Selfie Segmentation model if not present
import os
if not os.path.exists('mediapipe_selfie_segmentation.tflite'):
    print("Downloading MediaPipe Selfie Segmentation model...")
    !curl -O https://storage.googleapis.com/mediapipe-models/image_segmenter/selfie_segmenter/float/latest/selfie_segmenter.tflite
    !mv selfie_segmenter.tflite mediapipe_selfie_segmentation.tflite

BaseOptions = mp.tasks.BaseOptions
ImageSegmenter = mp.tasks.vision.ImageSegmenter
ImageSegmenterOptions = mp.tasks.vision.ImageSegmenterOptions
VisionRunningMode = mp.tasks.vision.RunningMode

options = ImageSegmenterOptions(
    base_options=BaseOptions(model_asset_path='mediapipe_selfie_segmentation.tflite'),
    running_mode=VisionRunningMode.IMAGE,
    output_category_mask=True
)

try:
    segmenter = ImageSegmenter.create_from_options(options)
    print("✅ MediaPipe Selfie Segmenter bereit")
except Exception as e:
    print(f"⚠️ Selfie Segmenter mit Fallback geladen: {e}")
    segmenter = None

# 4 | YOLO Object Detection

In [ ]:
def detect_objects_yolo(image_path, conf_threshold=0.5):
    """
    YOLO Objekterkennung mit Koordinaten.
    """
    image = cv2.imread(image_path)
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    
    results = yolo_model.predict(source=image_path, conf=conf_threshold, verbose=False)
    
    detections = []
    if results[0].boxes is not None:
        for box in results[0].boxes:
            x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().astype(int)
            conf = box.conf.cpu().numpy()[0]
            cls = int(box.cls.cpu().numpy()[0])
            class_name = yolo_model.names[cls]
            
            detections.append({
                'class_name': class_name,
                'confidence': conf,
                'bbox': (x1, y1, x2, y2),
                'center': ((x1 + x2) // 2, (y1 + y2) // 2)
            })
    
    return image_rgb, detections, results

test_image_path = "files/peoples.png"
image_rgb, detections, _ = detect_objects_yolo(test_image_path)

print(f"✅ {len(detections)} Objekte erkannt")
for i, det in enumerate(detections[:3], 1):
    print(f"  {i}. {det['class_name']} (Conf: {det['confidence']:.2f})")

# 5 | Hybrid Mask Creation: YOLO + MediaPipe Selfie Segmenter

In [ ]:
def create_mask_with_selfie_segmenter(image_rgb, bbox, object_class, use_selfie_segmenter=True):
    """
    Erstellt Maske mit MediaPipe Selfie Segmenter für Person-Objekte.
    Für andere Objekte: Fallback zur Bounding Box.
    """
    h, w = image_rgb.shape[:2]
    x1, y1, x2, y2 = bbox
    
    # Versuche Selfie Segmentation für Personen
    if use_selfie_segmenter and object_class.lower() == 'person' and segmenter is not None:
        print("  🎯 Verwende MediaPipe Selfie Segmenter für präzise Person-Maske...")
        try:
            # Crop zur Bounding Box
            cropped = image_rgb[y1:y2, x1:x2]
            cropped_h, cropped_w = cropped.shape[:2]
            
            # MediaPipe Segmentierung
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=cropped)
            segmentation_result = segmenter.segment(mp_image)
            
            # Maske extrahieren
            if hasattr(segmentation_result, 'category_mask'):
                mask_data = segmentation_result.category_mask.numpy_view()
            else:
                mask_data = segmentation_result.confidence_masks[0].numpy_view()
            
            # Normalisieren zu 0-255
            mask_cropped = (mask_data * 255).astype(np.uint8)
            
            # Auf Original-Größe resizen
            mask_cropped = cv2.resize(mask_cropped, (cropped_w, cropped_h), interpolation=cv2.INTER_LINEAR)
            
            # In Vollbild einplatzieren
            mask = np.zeros((h, w), dtype=np.uint8)
            mask[y1:y2, x1:x2] = mask_cropped
            
            print("    ✅ Selfie Segmenter erfolgreich angewendet")
            
        except Exception as e:
            print(f"    ⚠️ Selfie Segmenter Fehler: {e}. Nutze Fallback...")
            mask = create_rectangular_mask(h, w, bbox)
    else:
        # Fallback: Rechteckige Maske
        if object_class.lower() != 'person':
            print(f"  📦 Objekt-Klasse '{object_class}' nicht unterstützt. Nutze rechteckige Maske...")
        mask = create_rectangular_mask(h, w, bbox)
    
    # Morphologische Operationen
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    
    # Gaussian Blur für glatte Übergänge
    mask = cv2.GaussianBlur(mask, (7, 7), 0)
    
    return mask

def create_rectangular_mask(h, w, bbox):
    """
    Fallback: Rechteckige Maske aus Bounding Box.
    """
    mask = np.zeros((h, w), dtype=np.uint8)
    x1, y1, x2, y2 = bbox
    mask[y1:y2, x1:x2] = 255
    return mask

# Test mit bester Detection (Person)
best_detection = max(detections, key=lambda x: x['confidence'])
print(f"\n🎯 Beste Detection: {best_detection['class_name']}")
print(f"   Koordinaten: {best_detection['bbox']}")

mask = create_mask_with_selfie_segmenter(
    image_rgb, 
    best_detection['bbox'],
    best_detection['class_name'],
    use_selfie_segmenter=True
)
print("✅ Hybrid-Maske erstellt")

# 6 | Maske Visualisierung

In [ ]:
# Vergleich: Rechteck-Maske vs Selfie-Segmenter-Maske
rectangular_mask = create_rectangular_mask(image_rgb.shape[0], image_rgb.shape[1], best_detection['bbox'])

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Original
axes[0].imshow(image_rgb)
x1, y1, x2, y2 = best_detection['bbox']
rect = plt.Rectangle((x1, y1), x2-x1, y2-y1, linewidth=2, edgecolor='r', facecolor='none')
axes[0].add_patch(rect)
axes[0].set_title("Original mit YOLO BBox")
axes[0].axis('off')

# Rechteck-Maske (A2 Methode)
axes[1].imshow(rectangular_mask, cmap='gray')
axes[1].set_title("A2: Rechteck-Maske")
axes[1].axis('off')

# Selfie-Segmenter-Maske (A3 Methode)
axes[2].imshow(mask, cmap='gray')
axes[2].set_title("A3: MediaPipe Selfie-Maske")
axes[2].axis('off')

plt.suptitle("Vergleich: Mask Creation Methoden", fontsize=14, fontweight='bold')
plt.tight_layout()
os.makedirs('output', exist_ok=True)
plt.savefig('output/mask_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

print("✅ Masken verglichen")

# 7 | Comic-Effekt mit Präzisions-Maske

In [ ]:
def apply_comic_effect(image_rgb, mask=None, edge_threshold=50, num_colors=5):
    """
    Comic-Effekt mit Maske-Blending.
    """
    image_bgr = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2BGR)
    
    # Kanten erkennen
    gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)
    edges = cv2.Canny(gray, edge_threshold, edge_threshold * 2)
    edges = cv2.medianBlur(edges, 5)
    
    # Bilateral Filter
    smoothed = cv2.bilateralFilter(image_bgr, 9, 75, 75)
    
    # K-Means Clustering
    z = smoothed.reshape((-1, 3))
    z = np.float32(z)
    criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 10, 1.0)
    ret, labels, center = cv2.kmeans(z, num_colors, None, criteria, 10, cv2.KMEANS_RANDOM_CENTERS)
    center = np.uint8(center)
    res = center[labels.flatten()]
    posterized = res.reshape(smoothed.shape)
    
    # Kanten-Overlay
    edges_bgr = cv2.cvtColor(edges, cv2.COLOR_GRAY2BGR)
    edges_bgr = (edges_bgr > 0).astype(np.uint8) * 255
    comic = cv2.bitwise_and(posterized, cv2.bitwise_not(edges_bgr))
    comic = np.where(edges_bgr > 0, 0, comic)
    
    comic_rgb = cv2.cvtColor(comic, cv2.COLOR_BGR2RGB)
    
    # Weiches Blending mit normalisierter Maske
    if mask is not None:
        mask_normalized = (mask / 255.0).astype(np.float32)
        mask_3d = np.stack([mask_normalized, mask_normalized, mask_normalized], axis=2)
        comic_rgb = (comic_rgb * mask_3d + image_rgb * (1 - mask_3d)).astype(np.uint8)
    
    return comic_rgb, edges

# Comic-Effekt anwenden
comic_result, edges = apply_comic_effect(image_rgb, mask)
print("✅ Comic-Effekt angewendet")

# 8 | Finale Visualisierung

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Panel 1: Original mit BBox
axes[0, 0].imshow(image_rgb)
x1, y1, x2, y2 = best_detection['bbox']
rect = plt.Rectangle((x1, y1), x2-x1, y2-y1, linewidth=2, edgecolor='g', facecolor='none')
axes[0, 0].add_patch(rect)
axes[0, 0].set_title('1. YOLO Detection', fontsize=14, fontweight='bold')
axes[0, 0].axis('off')

# Panel 2: Selfie-Segmenter Maske
axes[0, 1].imshow(mask, cmap='gray')
axes[0, 1].set_title('2. MediaPipe Selfie Mask\n(Präzisions-Segmentierung)', fontsize=14, fontweight='bold')
axes[0, 1].axis('off')

# Panel 3: Kanten
axes[1, 0].imshow(edges, cmap='gray')
axes[1, 0].set_title('3. Edge Detection', fontsize=14, fontweight='bold')
axes[1, 0].axis('off')

# Panel 4: Comic Effect
axes[1, 1].imshow(comic_result)
axes[1, 1].set_title('4. Comic Effect\n(mit Selfie-Maske)', fontsize=14, fontweight='bold')
axes[1, 1].axis('off')

plt.suptitle('A3: YOLO + MediaPipe Selfie Segmenter → Comic Workflow', 
             fontsize=16, fontweight='bold', y=0.98)
plt.tight_layout()
plt.savefig('output/workflow_a3_complete.png', dpi=100, bbox_inches='tight')
plt.show()

cv2.imwrite('output/a3_comic_effect.png', cv2.cvtColor(comic_result, cv2.COLOR_RGB2BGR))
print("✅ Workflow Complete!")

# 9 | Test mit apfel.png (Fallback Test)

In [ ]:
print("\n" + "="*60)
print("Test mit apfel.png (Fallback: Rechteck-Maske)")
print("="*60 + "\n")

image_apple, detections_apple, _ = detect_objects_yolo('files/apfel.png')

if detections_apple:
    best_apple = max(detections_apple, key=lambda x: x['confidence'])
    print(f"Detected: {best_apple['class_name']}")
    
    mask_apple = create_mask_with_selfie_segmenter(
        image_apple,
        best_apple['bbox'],
        best_apple['class_name'],
        use_selfie_segmenter=True
    )
    
    comic_apple, _ = apply_comic_effect(image_apple, mask_apple)
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].imshow(image_apple)
    x1, y1, x2, y2 = best_apple['bbox']
    rect = plt.Rectangle((x1, y1), x2-x1, y2-y1, linewidth=2, edgecolor='r', facecolor='none')
    axes[0].add_patch(rect)
    axes[0].set_title(f"Detection: {best_apple['class_name']}")
    axes[0].axis('off')
    
    axes[1].imshow(comic_apple)
    axes[1].set_title("Comic Effect (Apple)")
    axes[1].axis('off')
    
    plt.tight_layout()
    plt.show()
    
    cv2.imwrite('output/a3_apple_comic.png', cv2.cvtColor(comic_apple, cv2.COLOR_RGB2BGR))
    print("✅ Apple Test Complete")

# 10 | Zusammenfassung: Was ist neu in A3?

## 🎯 Hybrid-Workflow: YOLO + MediaPipe Selfie Segmenter

### ✨ Neuerungen in A3 vs A2

#### A2: Reine Bounding-Box Maske
```
Rechteckige Maske (7x7 Kernel + Gaussian Blur)
    ↓
    Zu simpel für komplexe Formen
```

#### A3: Hybrid Präzisions-Maske ⭐
```
YOLO Koordinaten (x1, y1, x2, y2)
    ↓
    Crop zur Bounding Box
    ↓
    MediaPipe Selfie Segmenter (Pixel-genau!)
    ↓
    Morphologische Operationen + Gaussian Blur
    ↓
    Perfekte Person-Maske
```

### 🔑 Schlüssel-Unterschiede

| Aspekt | A2 | A3 |
|--------|----|----|  
| **Maske-Typ** | Rechteck | Pixel-genau |
| **Person-Details** | Eckig | Abgerundet |
| **Hintergründe** | Ignoriert | Genau erkannt |
| **Qualität** | 🟡 Gut | ✅ Ausgezeichnet |
| **Für Personen** | OK | OPTIMAL |
| **Für Objekte** | Gut | Fallback zu Rechteck |

### 🎨 Resultat
- **A2**: Scharfe Ecken beim Comic-Effekt
- **A3**: Sanfte, natürliche Übergänge ⭐

### 💡 Intelligente Fallback-Logik
```python
if object_class == 'person' and segmenter available:
    # Verwende MediaPipe Selfie Segmenter
    mask = selfie_segmentation()
else:
    # Fallback zu Rechteck-Maske
    mask = rectangular_mask()
```

---

## 🏆 Produktions-Workflow

**A3 ist jetzt die beste Wahl für:**
- ✅ Person-basierte Anwendungen (Porträts, Avatare, etc.)
- ✅ Hochwertige Comic-Filter
- ✅ Professionelle Bild-Bearbeitung
- ✅ Kreative Hybrid-Systeme